# Notebook 05 of 7 — What-If + Attribution + Paper

*Portfolio Intelligence Engine — User Guide Series.*
[Series README](README.md) · [Story Bible](STORY_BIBLE.md) · Filed under
epic [#1352](https://github.com/prajoria/OpenBB/issues/1352).

---

## Where we are in Sam's story

NB04 surfaced three names that look wrong. My instinct is to add to NVDA (it's up, so it's working, right?), trim VNQ (real estate has been dead), and close AMD (hostile calendar). Before I do any of that with real money — I'm going to diff the portfolio, decompose where my returns actually came from, and paper-trade the revised plan.

By the end of this notebook we will be able to answer one question:

> *What would my three intuitive trades actually do to my book, and did I decide them for the right reasons?*


In [ ]:
# [CODE PLACEHOLDER — Phase B] environment sanity — assert .venv_portfolio is active; STATE dir created; friendly halt with setup command if not


## 1. Load state from NB03 + NB04

Basket from NB03. Events + smart-money artifacts from NB04. If they
don't exist (running NB05 standalone), regenerate the basket from the
locked list and stub the signals with a canned fallback so the diff
still runs.

*The code cell below loads all three artifacts.*

In [ ]:
# [CODE PLACEHOLDER — Phase B] load basket.json + xray.pkl + smart_money.pkl; regenerate any missing artifact from fallback fixtures; print load report


## 2. The three candidate trades — in English first

Before we touch the diff engine, write out what we intend to do in plain
words. Doing this by hand every time is a discipline; if I can't say
in one sentence why I'm placing the trade, I have no business placing
it.

Sam's three:

1. **Add to NVDA** — up on the year, positive smart-money in NB04.
2. **Trim VNQ** — REITs weak, rate outlook hostile.
3. **Close AMD** — earnings in 5 days, negative smart-money, holding
   into it feels wrong.

Write them down. We'll come back to this list.

*The code cell below encodes the three trades as a list of dicts with a
`rationale` field per trade — the same shape NB05 §6 will submit as
paper orders.*

In [ ]:
# [CODE PLACEHOLDER — Phase B] encode Sam's 3 candidate trades as list[dict] with (ticker, action, delta_weight, rationale); pretty-print the list


## 3. What-if diff — PR #905

`obb.portfolio_intel` ships a stateless what-if engine (shipped in PR
#905). Give it a basket + a list of proposed trades, and it returns a
side-by-side before/after on every metric NB03 introduced: HHI, sector
weights, Sharpe, tracking error, MaxDD, top-K concentration,
single-name kill-shot.

If a trade looks *intuitively* like risk reduction but the diff
shows concentration going *up*, that's the tool catching you.

*The code cell below runs the what-if engine on Sam's three trades and
renders the before/after table with deltas highlighted.*

In [ ]:
# [CODE PLACEHOLDER — Phase B] call the what-if diff engine (portfolio_intel router) with basket + trades; render before/after table with signed deltas


## 4. Reading a what-if

Which deltas matter, which are noise:

- **HHI up + top-1 weight up** → you concentrated. That's usually
  wrong unless the smart-money signal on that name is overwhelming.
- **HHI up + top-1 weight down** → you spread the concentration across
  a few names. Sometimes intended, sometimes accidental.
- **Sharpe delta > +0.1** → suspicious on a small trade. What's
  giving you that much improvement from one trade?
- **MaxDD delta > +2%** → you added tail risk. Was that on purpose?

Sam's trades: the "add NVDA + trim VNQ" moves NB03's already-scary
Tech overweight *further up*. The what-if numbers should reflect
that. The "close AMD" helps sector diversity but the size is small so
the effect is muted.

*The code cell below prints a short interpretation for each row of the
diff table, in Sam's voice.*

In [ ]:
# [CODE PLACEHOLDER — Phase B] for each delta row, print a one-sentence Sam-voice interpretation (concentration up vs down, sharpe suspicious, etc.)


## 5. Brinson attribution — where did my returns come from?

Separate from the what-if: **why has my basket underperformed?** The
Brinson-Fachler decomposition against SPY splits my active return
into:

- **Allocation** — did I over/under-weight sectors that mattered?
- **Selection** — inside each sector, did I pick better or worse names
  than the benchmark?

The uncomfortable answer for most retail books: **allocation** drives
it, not **selection**. Being underweight tech in a tech-up year hurts
more than picking the "wrong" tech name. If my alpha is negative and
allocation is the dominant contributor, I was betting on
sector-timing without knowing it.

*The code cell below runs Brinson-Fachler on the basket vs SPY over
the last 12 months and prints the allocation-vs-selection decomposition
by sector.*

In [ ]:
# [CODE PLACEHOLDER — Phase B] run Brinson-Fachler on basket vs SPY, 12-month lookback; render decomposition table (sector, allocation contrib, selection contrib)


## 6. Paper blotter — placing the trades without capital

`openbb_portfolio_intel/paper/` is the paper-trading engine. Every
paper order carries a **mandatory rationale field** — a discipline the
system enforces to make sure I can never claim later "I don't remember
why I bought this." NB07's Monday-morning routine picks up the blotter
state.

*The code cell below submits Sam's three trades to the paper blotter
with their rationales, marks the fills at current mid-quote, and
renders the blotter state.*

In [ ]:
# [CODE PLACEHOLDER — Phase B] submit 3 paper trades via portfolio_intel/paper; mark-to-market at current quote; render blotter table


## 7. Low-BP alert — deliberately trigger one

The paper engine emits `Alert` objects on portfolio conditions: a
low-buying-power state, a GTC order about to expire, a stop level
approaching. To see the shape without waiting for a real event, we
submit an over-sized order that will not fit under simulated buying
power.

*The code cell below submits an over-sized order, catches the alert,
and prints its full `Alert` shape.*

In [ ]:
# [CODE PLACEHOLDER — Phase B] submit intentionally over-sized paper order; capture the raised Alert object; pretty-print its fields


## 8. Save state for NB07

`.notebook_state/paper_blotter.pkl` — the blotter + the alert object.
NB07's Monday-morning routine reloads it.

*The code cell below pickles the blotter state.*

In [ ]:
# [CODE PLACEHOLDER — Phase B] pickle blotter state + alerts to .notebook_state/paper_blotter.pkl


---

## What is NOT in this notebook

- **Live broker adapter.** Paper only. The `execution/` module has the shape a real broker adapter would slot into; not shipped.
- **Margin-call simulation.** The buying-power model is single-account, no margin math yet.
- **Overnight risk report.** The `Alert` engine covers today; overnight-gap risk projection is future work.

## Preview of NB06

The paper trade is on. But NB05 was one moment — one basket, one set of signals, one diff. The strategy behind it — 'rebalance to top-5 owner-earnings yield inside the basket every month' — hasn't been tested against history. In NB06 I turn it into a `BacktestConfig` and see whether the underlying idea has real edge, or whether NB05 was just one lucky what-if.
